# Step 12 Monitor Inspection

Use this notebook to run one monitored request or inspect a monitor JSONL file from a CLI/live-test run. The runtime agents see only `prompts/` content; this notebook is for human inspection after the run.

In [ ]:
from datetime import datetime
import logging
from pathlib import Path
import sys
from IPython.display import Markdown, display
from dotenv import load_dotenv

load_dotenv()

for noisy_logger in ("arxiv", "httpcore", "httpx"):
    logging.getLogger(noisy_logger).setLevel(logging.WARNING)

from auto_research import run_research
from auto_research.monitor import RunMonitor
from auto_research.schemas import ResearchRequest

## Run With Live Chatter

`chatter=7` prints compact live monitor lines while the run is active. The JSONL log still receives every event, including level-10 model-call details.

In [ ]:
request = ResearchRequest(
    question=(
            "IXPE has performed a Cyg X-1 observation in its soft state on 2023-05-24. "
            "Find simultaneous or near-simultaneous observations from other X-ray observatories "
            "(e.g. NICER, NuSTAR, XMM-Newton) and perform a joint broadband spectral fit "
            "for this epoch. epoch including observations: IXPE: '02008401', NICER: '6643010101' and '6643010102', NuSTAR: '80902318002'."
        ),
        context=(
            "TEST SESSION — evaluation criteria are relaxed as follows:\n"
            "- Absent NuSTAR and NICER backgrounds are acceptable and expected: "
            "Cyg X-1 is an extremely bright source for which background files are not used "
            "(background contribution is negligible; this is standard practice for bright sources).\n"
            "- The primary goal is to verify that analysts correctly invoked the available "
            "toolkit tools (load_data, fit_tbfeo_hss, run_fit, xspec_status, plot_fit, "
            "plot_iqu, group_pha) and that any inline Python executed via "
            "`mcp.builtin.bash_exec` with `python -c` successfully imports xspec and "
            "xspec_utils without errors.\n"
            "- The existing toolkit will not produce publication-level results. "
            "You may send back one round of revision to analysts, asking for better results. "
            "They should attempt to run inline calculations via python -c until they get "
            "non-error outputs and reasonable-looking fit statistics, "
            "but they are not expected to achieve publication-quality fits.\n"
            "- Accept the analysis if tools were called with correct arguments and returned "
            "non-error outputs, even if fit statistics are not publication-quality.\n"
            "- If any of the tools or environments seemed broken, abort the run immediately and report."
        )
)

LOG_DIR = Path("/data/fzhou/auto_research/logs/dev/step-12")
LOG_DIR.mkdir(parents=True, exist_ok=True)
monitor_jsonl = LOG_DIR / f"cyg_x1_ixpe_softstate_{datetime.now().strftime('%Y%m%dT%H%M%S')}_monitor.jsonl"
monitor = RunMonitor(chatter=7, jsonl_path=monitor_jsonl, stream=sys.stdout)
run = await run_research(request, monitor=monitor)
run.monitor.write_markdown(monitor_jsonl.with_suffix(".md"))
monitor_jsonl, monitor_jsonl.with_suffix(".md"), run.verdict

In [ ]:
display(Markdown(run.markdown()))

## Inspect An Existing Monitor

In [ ]:
# Inspect the most recent monitor log from logs/dev/step-12/.
# Override existing_jsonl to point at a specific file if needed.
log_dir = Path("/data/fzhou/auto_research/logs/dev/step-12")
candidates = sorted(log_dir.glob("*.jsonl")) if log_dir.exists() else []
existing_jsonl = candidates[-1] if candidates else None

if existing_jsonl:
    rows = [l for l in existing_jsonl.read_text(encoding="utf-8").splitlines() if l.strip()]
    print(f"{len(rows)} monitor events in {existing_jsonl}")
    md = existing_jsonl.with_suffix(".md")
    if md.exists():
        display(Markdown(md.read_text(encoding="utf-8")))
else:
    print(f"No logs yet in {log_dir}")